# Badminton singles analysis — Milestone 1 on Google Colab

Runs the same pipeline as the local machine (`python -m badminton_analysis ...`) with
`configs/colab.yaml`, which differs from `configs/default.yaml` only in its paths.

**Google Drive layout expected** (`My Drive/badminton/`):
```
data/BFMD_data/annotations/<type>/<match>.json   BFMD annotations (folder tree)
videos/<match>.mp4                               broadcast videos (optional, for frame overlays)
code/                                            the repo, only if CODE_SOURCE = "drive"
outputs/                                         written by this notebook
```
A GPU is **not** needed for Tasks 1–3 (they run on CPU in a few minutes). Results are
written to `My Drive/badminton/outputs/` so they survive the session.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/badminton'
!ls -la "$DRIVE"

## 2. Get the code
Choose one source. `github` needs the `milestone1` branch pushed (and a token if the repo is private);
`drive` copies `My Drive/badminton/code/`.

In [ ]:
CODE_SOURCE = "github"         # "github" (public repo) or "drive"
GITHUB_URL = "https://github.com/bzaat/Badminton_Project.git"
BRANCH = "main"

import os, shutil, subprocess
REPO = '/content/repo'
os.chdir('/content')                 # never delete the directory we are standing in (re-runs)
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if CODE_SOURCE == "github":
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, GITHUB_URL, REPO], check=True)
else:
    shutil.copytree(f'{DRIVE}/code', REPO)
os.chdir(REPO)
assert os.path.exists(f'{REPO}/badminton_analysis/__main__.py'), 'code checkout failed'
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout or os.listdir(REPO))

## 3. Copy the BFMD annotations to local disk (reading 1.8 GB of JSON from Drive directly is slow)

In [ ]:
%%bash
set -euo pipefail
rm -rf /content/BFMD_data
cp -r /content/drive/MyDrive/badminton/data/BFMD_data /content/BFMD_data
du -sh /content/BFMD_data
for d in metadata hit_inferred shot_type pose shuttle player_bbox court court_perframe; do
  n=$(ls /content/BFMD_data/annotations/$d | wc -l); echo "$d: $n files"
  [ "$n" -ge 12 ] || { echo "ERROR: $d has only $n files - Drive upload incomplete?"; exit 1; }
done

## 4. Dependencies
Colab already ships numpy, scipy, pandas, scikit-learn, matplotlib, opencv and PyYAML; only
pytest is installed (with uv, as in the project). Versions are printed for the record.

In [ ]:
!pip -q install uv && uv pip install --system -q pytest
import sys, numpy, scipy, sklearn, pandas, matplotlib, cv2, yaml
print(sys.version.split()[0], 'numpy', numpy.__version__, 'scipy', scipy.__version__, 'sklearn', sklearn.__version__,
      'pandas', pandas.__version__, 'matplotlib', matplotlib.__version__, 'opencv', cv2.__version__, 'yaml', yaml.__version__)
!nvidia-smi -L || echo "no GPU (not needed for Tasks 1-4)"

## 5. Tests (synthetic + real-data regression tests)

In [ ]:
%%bash
set -euo pipefail
cd /content/repo
BFMD_ROOT=/content/BFMD_data python -m pytest -q -x 2>&1 | tail -25

## 6. Run the pipeline

In [ ]:
%%bash
# one command = build, validate, viz-samples, task2, task3, task4 (~8 min on CPU).
# %%bash + set -e: if a step fails, this cell fails and "Run all" stops here with the error.
set -euo pipefail
cd /content/repo
python -m badminton_analysis --config configs/colab.yaml all 2>&1 | tail -40

## 7. Results

In [ ]:
from IPython.display import Markdown, Image, display
OUT = f'{DRIVE}/outputs'
import os
missing = [t for t in ('task2', 'task3', 'task4') if not os.path.exists(f'{OUT}/{t}/report.md')]
assert not missing, f"no report for {missing}: the run cell above did not finish - see its error output"
for t in ('task2', 'task3', 'task4'):
    display(Markdown(open(f'{OUT}/{t}/report.md').read()))
display(Image(f'{OUT}/task2/task2_summary.png'))
display(Image(f'{OUT}/task3/confusion_fine.png'))
display(Image(f'{OUT}/task4/occupancy_heatmaps.png'))

## 8. Optional: annotation overlay on a real video frame
Needs the match video in `My Drive/badminton/videos/`.

In [ ]:
import sys; sys.path.insert(0, '/content/repo')
import numpy as np, cv2, glob, os
from badminton_analysis.config import load_config
from badminton_analysis.build import load_match
from badminton_analysis.viz import read_frames, draw_frame_overlay
cfg = load_config('configs/colab.yaml')
videos = sorted(glob.glob(f'{cfg.video_root}/*.mp4'))
print(videos)
if videos:
    match = os.path.basename(videos[0])[:-4]
    md_ = load_match(cfg, match)
    frame = md_.hits[100]['frame']
    img = draw_frame_overlay(read_frames(videos[0], [frame])[frame], md_, frame)
    out = f'{cfg.output_dir}/overlay_example.jpg'; cv2.imwrite(out, img)
    display(Image(out))